In [270]:
import os
import json
import requests

import pandas as pd
import dotenv
import redis
import numpy as np

In [271]:
# open .env file and get API keys
env_path = os.path.abspath('../.env.development.local')
dotenv.load_dotenv(env_path)
KV_REST_API_READ_ONLY_TOKEN = os.getenv("KV_REST_API_READ_ONLY_TOKEN")
KV_REST_API_TOKEN = os.getenv("KV_REST_API_TOKEN")
KV_REST_API_URL = os.getenv("KV_REST_API_URL")
KV_URL = os.getenv("KV_URL")

# Set headers for authentication
headers = {
    "Authorization": f"Bearer {KV_REST_API_TOKEN}",
    "Content-Type": "application/json"
}

In [272]:
# Adjust url to work with redis
redis_url = KV_URL
if redis_url.startswith("redis://"):
    redis_url = 'rediss://' + redis_url[len('redis://'):]
r = redis.from_url(redis_url)

In [95]:
# Import datasets
# datasets = {}
# dataset_names = ['clfever', 'phemeplus', 'vitc']
# for dataset_name in dataset_names:
#     with open(f'{dataset_name}.json') as f:
#         datasets[dataset_name] = json.load(f)

In [144]:
# Import VITC daraset
with open("vitc_evaluation_sup_ref.json") as f:
    vitc = json.load(f)

# Test labels
labels = []
for claim in vitc:
    labels.append(claim['label'])
labels = np.array(labels)
np.unique(labels, return_counts=True)

(array(['REFUTES', 'SUPPORTS'], dtype='<U8'), array([217, 283]))

In [165]:
vitc_dict = {}
for datapoint in vitc:
    vitc_dict[datapoint['claim_id']] = datapoint

In [130]:
# Create batches of 25 claims
vitc_batches = {}
for i in range(0, len(vitc), 25):
    batch_dataset = vitc[i:i+25]
    claim_ids = []
    for batch_datapoint in batch_dataset:
        id = batch_datapoint['claim_id']
        claim_ids.append(id)
    batch_id = f'batch_vitc_{i//25 + 1}'
    vitc_batches[batch_id] = claim_ids

# Create an extra 8 batches to confirm the annotations
# confirm_batches = {f'batch_vitc_confirm_{i}': [] for i in range(1, 5)}

# for i in range(1,5):
    
# for batch_id in batches.keys():
#     claim_ids = batches[batch_id]
#     for i in range(1,5):
#         confirm_batch_id = f'batch_vitc_confirm_{i}'
#         confirm_batches[confirm_batch_id].append(claim_ids[8 * (i-1)])


In [145]:
# Populate Vercel KV with vitc datasets
# for datapoint in vitc:
#     id = datapoint['claim_id']
#     r.hset(id, mapping={
#         'claim': datapoint['claim'],
#         'evidence': datapoint['evidence'],
#         'label': datapoint['label']
#     })

# Populate Vercel KV with vitc batches
for batch_id in vitc_batches.keys():
    claim_ids = vitc_batches[batch_id]
    r.hset(batch_id, mapping={
        'claim_ids': json.dumps(claim_ids)
    })

# Populate Vercel KV with confirm batches



In [96]:
# Populate Vercel KV with datasets
# for dataset_name in dataset_names:
#     dataset = datasets[dataset_name]
#     for datapoint in dataset:
#         id = datapoint['claim_id']
        # r.hset(id, mapping={
        #     'claim': datapoint['claim'],
        #     'evidence': datapoint['evidence'],
        #     'label': datapoint['label']
        # })

In [54]:
# Create batches containing 25 datapoints each
# batch_ids = []
# for dataset_name in dataset_names:
#     dataset = datasets[dataset_name]
#     for i in range(0, len(dataset), 25):
#         batch_dataset = dataset[i:i+25]
#         claim_ids = []
#         for batch_datapoint in batch_dataset:
#             id = batch_datapoint['claim_id']
#             claim_ids.append(id)
#         batch_id = f'batch_{dataset_name}_{i//25 + 1}'
#         batch_ids.append(batch_id)
#         r.hset(batch_id, mapping={'claim_ids': json.dumps(claim_ids)})   

In [107]:
# Import assessesment samples
vitc_assessment = pd.read_csv('Assesment samples - VitC.csv')

In [28]:
vitc_assessment

,Id,claim,evidence,Ground Truth,Label,Unnamed: 5
0,asses_1_vitc,There have been more than five confirmed cases...,"On 25 January 2020 , the number of laboratory-...",SUPPORTS,d,NaN
1,asses_2_vitc,The most prominent smartphone vendor in the wo...,BlackBerry was one of the most prominent smart...,SUPPORTS,a,This one is in the guideline
2,asses_3_vitc,"According to expectations , the Boeing B-52 St...","After being upgraded between 2013 and 2015 , i...",REFUTES,d,NaN
3,asses_4_vitc,Marcus Bentley is a British chef .,"Marcus Morgan Bentley ( born October 4 , 1967 ...",SUPPORTS,d,NaN
4,asses_5_vitc,"Before March 29 , 2020 , Nevada had less than ...","As of March 29 , 2020 , 738 positive cases and...",REFUTES,a,NaN


In [30]:
# iterate through rows of dataframe
assess_ids = []
for index, row in vitc_assessment.iterrows():
    id = row["Id"]
    assess_ids.append(id)
    claim = row["claim"]
    evidence = row["evidence"]
    label = row["Ground Truth"]
    reasoning = row["Label"]
    if reasoning == 'a':
        reasoning = 'abductive'
    elif reasoning == 'd':
        reasoning = 'deductive'
    
    r.hset(id, mapping={
        'claim': claim,
        'evidence': evidence,
        'label': label,
        'reasoning': reasoning
    })


In [108]:
# create batch for assessement
r.hset('assess_vitc', mapping={'claim_ids': json.dumps(assess_ids)}) 

ConnectionError: Error 54 connecting to model-bedbug-21479.upstash.io:6379. Connection reset by peer.

In [34]:
r.hgetall('assess_vitc')

{b'claim_ids': b'["asses_1_vitc", "asses_2_vitc", "asses_3_vitc", "asses_4_vitc", "asses_5_vitc"]'}

In [257]:
# Get list of all submissions
r.lrange('participants',0,-1)

[b'{"participant":"665a9ddb536966183500a7b1","batchId":"batch_vitc_6","stage":"annotation"}',
 b'{"participant":"65e68fb1a21db3bbe1cfd991","batchId":"batch_vitc_8","stage":"annotation"}',
 b'{"participant":"66f5c81fa771db6db1bc2382","batchId":"batch_vitc_14","stage":"annotation"}',
 b'{"participant":"665a9ddb536966183500a7b1","batchId":"batch_vitc_6","stage":"assessment"}',
 b'{"participant":"6685533ec290d5a299807e10","batchId":"batch_vitc_10","stage":"annotation"}',
 b'{"participant":"6680452004dea60c1f984bba","batchId":"batch_vitc_13","stage":"assessment"}',
 b'{"participant":"66f5c81fa771db6db1bc2382","batchId":"batch_vitc_14","stage":"assessment"}',
 b'{"participant":"6685533ec290d5a299807e10","batchId":"batch_vitc_10","stage":"assessment"}',
 b'{"participant":"66c396ad1fd21de8d95c3921","batchId":"batch_vitc_6","stage":"assessment"}',
 b'{"participant":"66da0e619af6e27e9e82bc75","batchId":"batch_vitc_11","stage":"annotation"}',
 b'{"participant":"6314cea1992c8f31e0bd42db","batchId"

In [274]:
# get ids of participants who completed the task
completed_batches = []
completed_participants = []
submissions = r.lrange('participants',0,-26)
for submission in submissions:
    # convert to dictionary from bytes
    submission = submission.decode('utf-8')
    submission = json.loads(submission)

    if submission["stage"] == "annotation":
        completed_batches.append(submission["batchId"])
        completed_participants.append(submission["participant"])

In [280]:
len(completed_batches)

15

In [275]:
np.unique(completed_batches, return_counts=True)

(array(['batch_vitc_10', 'batch_vitc_11', 'batch_vitc_12', 'batch_vitc_14',
        'batch_vitc_16', 'batch_vitc_17', 'batch_vitc_18', 'batch_vitc_19',
        'batch_vitc_2', 'batch_vitc_3', 'batch_vitc_4', 'batch_vitc_5',
        'batch_vitc_6', 'batch_vitc_8', 'batch_vitc_9'], dtype='<U13'),
 array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]))

In [276]:
vitc_queue = [id for id in vitc_batches.keys() if id not in completed_batches]
vitc_queue

['batch_vitc_1',
 'batch_vitc_7',
 'batch_vitc_13',
 'batch_vitc_15',
 'batch_vitc_20']

In [261]:
# delete queue and then only add remaining batches
r.delete('queue')
r.lpush('queue', *vitc_queue)

5

In [262]:
r.rpush('queue', *["batch_vitc_1", "batch_vitc_2", "batch_vitc_3", "batch_vitc_4"])

9

In [273]:
queue = r.lrange('queue', 0, -1)
print(len(queue))
print(queue)

9
[b'batch_vitc_20', b'batch_vitc_15', b'batch_vitc_13', b'batch_vitc_7', b'batch_vitc_1', b'batch_vitc_1', b'batch_vitc_2', b'batch_vitc_3', b'batch_vitc_4']


In [256]:
def convert_from_byte(byte_dict):
    return {key.decode('utf-8'): value.decode('utf-8') for key, value in byte_dict.items()}

In [277]:
dataset = []
for participant in completed_participants:
    submission = r.hgetall(participant)
    submission = convert_from_byte(submission)
    for key in submission:
        if "asses" not in key:
            data = vitc_dict[key]
            data['reasoning'] = submission[key]
            data['participant'] = participant
            dataset.append(data)


In [278]:
dataset = pd.DataFrame(dataset)
dataset.to_json('dataset.json', orient='records', lines=True)

In [279]:
dataset['reasoning'].value_counts()

reasoning
deductive    231
abductive    144
Name: count, dtype: int64